# Feature Engineering v3 — RGB + AVG + NTSC Combination

Concatenate aligned 512-dimensional embeddings from the three selected v3 CNNs. All three v3 feature archives are required. This notebook prepares data; it does not train a model or calculate test accuracy.


## 1. Import dan Konfigurasi

In [1]:
import os
import uuid
import numpy as np

SEED = 30092026

RGB_PATH = 'features/rgb_features_v3.npz'
AVG_PATH = 'features/avg_features_v3.npz'
NTSC_PATH = 'features/ntsc_features_v3.npz'

OUTPUT_PATH = 'feature-engineering/combined_features_v3.npz'

SOURCE_ARCHIVES = [RGB_PATH, AVG_PATH, NTSC_PATH]
for path in SOURCE_ARCHIVES:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing v3 feature archive: {path}. Run all three v3 Training and Extraction notebooks first.')
print('Sources:', SOURCE_ARCHIVES)
print('Output:', OUTPUT_PATH)


Sources: ['features/rgb_features_v3.npz', 'features/avg_features_v3.npz', 'features/ntsc_features_v3.npz']
Output: feature-engineering/combined_features_v3.npz


## 2. Load Feature Archives

In [2]:
rgb = np.load(RGB_PATH, allow_pickle=False)
avg = np.load(AVG_PATH, allow_pickle=False)
ntsc = np.load(NTSC_PATH, allow_pickle=False)

required_keys = {
    "X_train_features",
    "X_val_features",
    "X_test_features",
    "y_train",
    "y_val",
    "y_test",
    "train_indices",
    "val_indices",
    "seed",
    "feature_dim",
    "feature_layer",
    "representation",
}

for name, archive in [("RGB", rgb), ("AVG", avg), ("NTSC", ntsc)]:
    missing = required_keys.difference(archive.files)
    if missing:
        raise KeyError(f"{name} archive kehilangan key: {sorted(missing)}")

    assert archive["representation"].item() == name
    assert archive["feature_layer"].item() == "svm_features"
    assert int(archive["feature_dim"].item()) == 512

    print(
        f"{name}:",
        archive["representation"].item(),
        "| feature_dim =",
        int(archive["feature_dim"].item()),
        "| seed =",
        int(archive["seed"].item())
    )

RGB: RGB | feature_dim = 512 | seed = 30092026
AVG: AVG | feature_dim = 512 | seed = 30092026
NTSC: NTSC | feature_dim = 512 | seed = 30092026


## 3. Verifikasi Sample Alignment

`RGB[i]`, `AVG[i]`, dan `NTSC[i]` harus berasal dari image CIFAR-10 yang sama.

In [3]:
assert int(rgb["seed"].item()) == SEED
assert int(avg["seed"].item()) == SEED
assert int(ntsc["seed"].item()) == SEED

assert np.array_equal(rgb["train_indices"], avg["train_indices"])
assert np.array_equal(rgb["train_indices"], ntsc["train_indices"])

assert np.array_equal(rgb["val_indices"], avg["val_indices"])
assert np.array_equal(rgb["val_indices"], ntsc["val_indices"])

for split in ["train", "val", "test"]:
    key = f"y_{split}"
    assert np.array_equal(rgb[key], avg[key])
    assert np.array_equal(rgb[key], ntsc[key])

print("PASS — seed sama.")
print("PASS — train_indices sama.")
print("PASS — val_indices sama.")
print("PASS — labels train/val/test sama.")
print("Feature aman untuk dikombinasikan.")

PASS — seed sama.
PASS — train_indices sama.
PASS — val_indices sama.
PASS — labels train/val/test sama.
Feature aman untuk dikombinasikan.


## 4. Periksa Shape Feature

In [4]:
for split in ["train", "val", "test"]:
    key = f"X_{split}_features"

    print(f"\n{split.upper()}")
    print("RGB :", rgb[key].shape)
    print("AVG :", avg[key].shape)
    print("NTSC:", ntsc[key].shape)

    assert rgb[key].shape[0] == avg[key].shape[0] == ntsc[key].shape[0]
    assert rgb[key].shape[1] == 512
    assert avg[key].shape[1] == 512
    assert ntsc[key].shape[1] == 512


TRAIN
RGB : (40000, 512)
AVG : (40000, 512)
NTSC: (40000, 512)

VAL
RGB : (10000, 512)
AVG : (10000, 512)
NTSC: (10000, 512)

TEST
RGB : (10000, 512)
AVG : (10000, 512)
NTSC: (10000, 512)


## 5. Concatenate RGB + AVG + NTSC

In [5]:
X_train_combined = np.concatenate(
    [
        rgb["X_train_features"],
        avg["X_train_features"],
        ntsc["X_train_features"],
    ],
    axis=1
).astype(np.float32, copy=False)

X_val_combined = np.concatenate(
    [
        rgb["X_val_features"],
        avg["X_val_features"],
        ntsc["X_val_features"],
    ],
    axis=1
).astype(np.float32, copy=False)

X_test_combined = np.concatenate(
    [
        rgb["X_test_features"],
        avg["X_test_features"],
        ntsc["X_test_features"],
    ],
    axis=1
).astype(np.float32, copy=False)

y_train = rgb["y_train"].astype(np.int64, copy=False)
y_val = rgb["y_val"].astype(np.int64, copy=False)
y_test = rgb["y_test"].astype(np.int64, copy=False)

print("Combined train:", X_train_combined.shape)
print("Combined val  :", X_val_combined.shape)
print("Combined test :", X_test_combined.shape)

assert X_train_combined.shape == (40000, 1536)
assert X_val_combined.shape == (10000, 1536)
assert X_test_combined.shape == (10000, 1536)

assert np.isfinite(X_train_combined).all()
assert np.isfinite(X_val_combined).all()
assert np.isfinite(X_test_combined).all()

Combined train: (40000, 1536)
Combined val  : (10000, 1536)
Combined test : (10000, 1536)


## 6. Feature Block Metadata

In [6]:
feature_dims = np.array([512, 512, 512], dtype=np.int64)
feature_starts = np.array([0, 512, 1024], dtype=np.int64)
feature_ends = np.array([512, 1024, 1536], dtype=np.int64)

print("Feature order :", ["RGB", "AVG", "NTSC"])
print("Feature dims  :", feature_dims)
print("Feature starts:", feature_starts)
print("Feature ends  :", feature_ends)

Feature order : ['RGB', 'AVG', 'NTSC']
Feature dims  : [512 512 512]
Feature starts: [   0  512 1024]
Feature ends  : [ 512 1024 1536]


## 7. Simpan `combined_features_v3.npz`

In [7]:
np.savez_compressed(
    OUTPUT_PATH,
    X_train_combined=X_train_combined,
    X_val_combined=X_val_combined,
    X_test_combined=X_test_combined,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    train_indices=rgb["train_indices"],
    val_indices=rgb["val_indices"],
    seed=np.array(SEED, dtype=np.int64),
    feature_order=np.array(["RGB", "AVG", "NTSC"]),
    feature_dims=feature_dims,
    feature_starts=feature_starts,
    feature_ends=feature_ends,
    total_feature_dim=np.array(1536, dtype=np.int64),
    source_archives=np.array(SOURCE_ARCHIVES),
    run_id=np.array(str(uuid.uuid4())),
)

print("Saved:", OUTPUT_PATH)

Saved: feature-engineering/combined_features_v3.npz


## 8. Verifikasi Output

In [8]:
with np.load(OUTPUT_PATH, allow_pickle=False) as combined:
    for key in combined.files:
        value = combined[key]
        if value.ndim == 0:
            print(f"- {key}: {value.item()}")
        else:
            print(f"- {key}: {value.shape}")

    assert combined["X_train_combined"].shape == (40000, 1536)
    assert combined["X_val_combined"].shape == (10000, 1536)
    assert combined["X_test_combined"].shape == (10000, 1536)

print()
print("PASS — combined_features_v3.npz valid.")

- X_train_combined: (40000, 1536)
- X_val_combined: (10000, 1536)
- X_test_combined: (10000, 1536)
- y_train: (40000,)
- y_val: (10000,)
- y_test: (10000,)
- train_indices: (40000,)
- val_indices: (10000,)
- seed: 30092026
- feature_order: (3,)
- feature_dims: (3,)
- feature_starts: (3,)
- feature_ends: (3,)
- total_feature_dim: 1536
- source_archives: (3,)
- run_id: db106007-b3e2-42c5-9aec-5682ee3ef231

PASS — combined_features_v3.npz valid.


# Combination complete

Output: `combined_features_v3.npz`. Run `Feature Engineering Training.ipynb` next. If any selected CNN changes, rerun its Extraction notebook and then this Combination notebook.
